In [1]:
import numpy as np
import pandas as pd
from charset_normalizer.cd import encoding_unicode_range
from sklearn import *
import sys
import os
#RFM(Recency, Frequency, Monetary)
# Sử dụng chính biến root_dir bạn vừa tìm được
root_path = '/mnt/d/datamining/Data-mining'

if root_path not in sys.path:
    sys.path.append(root_path)

# Bây giờ import sẽ hoạt động bình thường
from src.preprocessing import *

path = "/mnt/d/datamining/Data-mining/data/jewelry.csv"
column_names = [
    'event_time',    # Thời gian mua hàng
    'order_id',      # Mã đơn hàng
    'product_id',    # Mã sản phẩm
    'quantity',      # Số lượng
    'category_id',   # Mã danh mục
    'category_code', # Tên loại trang sức
    'brand_id',      # Mã thương hiệu
    'price',         # Giá tiền
    'user_id',       # Mã khách hàng
    'gender',        # Giới tính
    'color',         # Màu sắc (red, white, gold...)
    'metal',         # Chất liệu kim loại (gold, silver...)
    'gem'            # Loại đá quý (diamond, sapphire...)
]

# Ép về string
# tránh để string dài quá đọc csv sẽ tự chuyển sang int64
dtype_specs = {
    'order_id': str,
    'product_id': str,
    'category_id': str,
    'brand_id': str,
    'user_id': str,
    'category_code': str
}
dataset = pd.read_csv(path, header=None, names=column_names, dtype=dtype_specs)
dataset = run_phase_1_cleaning(dataset)

# Xem 5 dòng đầu tiên
print(dataset.head())

# Kiểm tra tổng quan: còn bao nhiêu dòng, cột, có còn giá trị null không?
print(dataset.info())

# Xem thống kê mô tả (giá tiền có gì bất thường không?)
print(dataset.describe())

                 event_time             order_id           product_id  \
0 2018-12-01 11:40:29+00:00  1924719191579951782  1842195256808833386   
1 2018-12-01 17:38:31+00:00  1924899396621697920  1806829193678291446   
2 2018-12-02 13:53:42+00:00  1925511016616034733  1842214461889315556   
3 2018-12-02 17:44:02+00:00  1925626951238681511  1835566849434059453   
4 2018-12-02 21:30:19+00:00  1925740842841014667  1873936840742928865   

   quantity          category_id category_code brand_id   price  \
0         1  1806829201890738522       earring        0  561.51   
1         1  1806829201848795479       unknown  unknown  212.14   
2         1  1806829201915904347       pendant        1   54.66   
3         1  1806829201915904347       pendant        0   88.90   
4         1  1806829201924292956      necklace        0  417.67   

               user_id   gender   color metal       gem  
0  1515915625207851155  unknown     red  gold   diamond  
1  1515915625071969944  unknown  yellow  g

In [2]:
def transform_for_association_rules(df: pd.DataFrame) -> pd.DataFrame:
    """
    TODO: Chuẩn bị dữ liệu cho bài toán Khai phá luật kết hợp (Association Rules).
        Yêu cầu:
        - Lọc các đơn hàng có quantity > 0.
        - Gom nhóm theo 'order_id'.
        - Output: Một DataFrame mà mỗi dòng là một đơn hàng, chứa list các sản phẩm (product_id hoặc category_code).
        VD: Order_1 -> ['Ring', 'Earring']
        earring: Bông tai.
        pendant: Mặt dây chuyền.
        necklace: Vòng cổ / Dây chuyền.
        ring: Nhẫn.
        bracelet: Vòng tay / Lắc tay.
        brooch: Ghim cài áo (xuất hiện ở dòng số 8 trong kết quả của bạn).
        unknown: Các sản phẩm chưa được phân loại rõ ràng trong hệ thống.
    """
    # Lọc các đơn hàng hợp lệ (quantity > 0) và category_code khác 'unknown'
    df_rules = df[(df['quantity'] > 0) & (df['category_code'] != 'unknown')].copy()
    # Gom nhóm theo order_id
    df_rules = df_rules.groupby('order_id')['category_code'].apply(lambda x: list(set(x))).reset_index()#  Bỏ reset_index() sẽ bị lỗi khi chạy apriori vì không còn là DataFrame nữa.
    # Đổi tên cột cho dễ hiểu
    df_rules.rename(columns={'category_code': 'item_list'}, inplace=True) # Không có inplace=True sẽ không đổi tên cột được
    return df_rules

def transform_for_user_profile(df: pd.DataFrame) -> pd.DataFrame:
    """
    TODO: Chuẩn bị dữ liệu cho bài toán Gom cụm khách hàng (Clustering) & Phân loại.
        Yêu cầu:
        - Gom nhóm theo 'user_id'.
        - Tính toán các chỉ số tổng hợp (RFM + Preferences):
            + total_spend (Sum price)
            + total_orders (Count unique order_id)
            + avg_order_value
            + recency (Số ngày từ lần mua cuối)
            + favorite_gem (Mode gem)
    """
    valid_df = df[
        (df['user_id'].notna()) & 
        (df['order_id'].notna()) & 
        (df['price'] > 0)
    ].copy()
    if valid_df.empty:
        return pd.DataFrame() # Trả về df rỗng nếu không có dữ liệu hợp lệ
    # Xác định mốc thời gian cuối cùng từ tập dữ liệu đã lọc
    last_time_in_data = valid_df['event_time'].max()
    # Định nghĩa hàm lấy danh sách Favorite Gems (Mode)
    def favorite_gem_list(x):
        counts = x.value_counts()
        if counts.empty: return ['unknown']
        max_count = counts.max()
        return counts[counts == max_count].index.tolist()
    # Gom nhóm và tính toán (Aggregation)
    agg_rules = {
        'price': 'sum',
        'order_id': 'nunique',
        'event_time': 'max',
        'gem': favorite_gem_list
    }
    df_users = valid_df.groupby('user_id').agg(agg_rules).reset_index()
    # Đổi tên và tính toán các chỉ số
    df_users.rename(columns={
        'price': 'total_spend',
        'order_id': 'total_orders',
        'event_time': 'last_purchase_date',
        'gem': 'favorite_gems'
    }, inplace=True)
    # Recency: Ngày cuối file - Ngày cuối của user
    df_users['recency'] = (last_time_in_data - df_users['last_purchase_date']).dt.days
    # Tổng chiTổng đơn
    df_users['avg_order_value'] = (df_users['total_spend'] / df_users['total_orders']).round(2)# Tiền sẵn sàng mua cho mỗi đơn hàng
    #Tổng đơn
    df_users['total_spend'] = df_users['total_spend'].round(2)
    # Xử lý an toàn cho Recency
    df_users['recency'] = df_users['recency'].clip(lower=0)# Đảm bảo không có giá trị âm do lệch thời gian
    # Loại bỏ cột ngày mua cuối vì mô hình đã có rency
    df_users = df_users.drop(columns=['last_purchase_date']) 
    return df_users

In [3]:
dataset_association = transform_for_association_rules(dataset)
print(dataset_association.head(20))


               order_id                item_list
0   1924719191579951782                [earring]
1   1925511016616034733                [pendant]
2   1925626951238681511                [pendant]
3   1925740842841014667               [necklace]
4   1925760595336888995                [earring]
5   1925764002260976330                [earring]
6   1926029494397698277                [pendant]
7   1926112416450478161  [brooch, ring, earring]
8   1926155622227640358                   [ring]
9   1926381405764321762                [earring]
10  1926724314904658447                   [ring]
11  1926785654931325894                [earring]
12  1926923883428970549                   [ring]
13  1927063562459546512                [earring]
14  1927142607515812568                [earring]
15  1927150150686343971                [pendant]
16  1927151232950993712               [bracelet]
17  1927494410593894972                [earring]
18  1927520374853992618                [earring]
19  1927527559520584

In [4]:
user_profile_df = transform_for_user_profile(dataset)
print(user_profile_df.head(20))



                user_id  total_spend  total_orders        favorite_gems  \
0   1313554333610017315       272.47             1            [unknown]   
1   1313555541586346734      3818.79             4            [diamond]   
2   1313565915903688919      2814.39             4      [topaz, fianit]   
3   1313582215799504936       130.00             1              [pearl]   
4   1313591033442861729      2067.09             6            [diamond]   
5   1313596598478963483        23.98             2            [unknown]   
6   1313605489329701533      5476.27            10  [diamond, sapphire]   
7   1313611771717616515       480.14             1            [diamond]   
8   1313634307679453761      1345.74             3       [mix, diamond]   
9   1313643326179639859      2266.57             4             [fianit]   
10  1313652173870989863       238.86             1            [unknown]   
11  1313658164888994672      4374.65             7   [fianit, sapphire]   
12  1313689435774124497  